In [1]:
!gcc --version
!nproc
!ls -lah /kaggle/input/

gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

4
total 12K
drwxr-xr-x 3 root root 4.0K Oct  9 07:36 .
drwxr-xr-x 5 root root 4.0K Oct  9 07:36 ..
drwxr-xr-x 3 root root 4.0K Oct  9 07:36 datasets


In [2]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    images = [
        f for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    if images:
        print("Folder:", root)
        print("Image count in folder:", len(images))
        print("Sample files:", images[:5])

Folder: /kaggle/input/datasets/xthink/coco-2017-val-images/val2017
Image count in folder: 5000
Sample files: ['000000011197.jpg', '000000219485.jpg', '000000151000.jpg', '000000371677.jpg', '000000038825.jpg']


In [3]:
%%writefile /kaggle/working/test_omp.c
#include <stdio.h>
#include <omp.h>

int main(void) {
    #pragma omp parallel
    {
        printf("Hello from thread %d\n", omp_get_thread_num());
    }
    return 0;
}

Writing /kaggle/working/test_omp.c


In [4]:
!gcc -fopenmp /kaggle/working/test_omp.c -o /kaggle/working/test_omp

In [5]:
!OMP_NUM_THREADS=4 /kaggle/working/test_omp

Hello from thread 0
Hello from thread 2
Hello from thread 3
Hello from thread 1


In [7]:
from pathlib import Path
from PIL import Image
import time

IMAGE_DIR = Path(
    "/kaggle/input/datasets/xthink/coco-2017-val-images/val2017"
)

WORK = Path("/kaggle/working/ssd_project")
PROCESSED = WORK / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

image_paths = sorted(IMAGE_DIR.glob("*.jpg"))

print("Input images:", len(image_paths))
assert len(image_paths) == 5000, "Expected 5000 COCO images"

start = time.perf_counter()
failed = []

for idx, path in enumerate(image_paths):
    try:
        with Image.open(path) as im:
            gray = im.convert("L").resize((256, 256))
            gray.save(PROCESSED / f"{idx:05d}.png")
    except Exception as e:
        failed.append((path.name, str(e)))

    if (idx + 1) % 500 == 0:
        print(f"Processed {idx + 1}/5000")

print(f"\nElapsed: {time.perf_counter() - start:.2f} seconds")
print("Successful:", 5000 - len(failed))
print("Failed:", len(failed))
print("Output folder:", PROCESSED)

Input images: 5000
Processed 500/5000
Processed 1000/5000
Processed 1500/5000
Processed 2000/5000
Processed 2500/5000
Processed 3000/5000
Processed 3500/5000
Processed 4000/5000
Processed 4500/5000
Processed 5000/5000

Elapsed: 157.27 seconds
Successful: 5000
Failed: 0
Output folder: /kaggle/working/ssd_project/processed


In [8]:
import os
from pathlib import Path
import subprocess

print("Input directory:", os.listdir("/kaggle/input"))
print("\nDataset mount details:")

result = subprocess.run(
    ["bash", "-lc", "mount | grep kaggle"],
    capture_output=True,
    text=True
)
print(result.stdout or "No Kaggle mounts listed")
print("\nInput directory details:")
!ls -la /kaggle/input
!ls -la /kaggle/input/datasets

Input directory: ['datasets']

Dataset mount details:
/tmp/fs/kaggle_YBCZdHDIZmW2XvYn0QjNsAjbAtYqr3QY2nZifZ60iVY-356678632-webtier/srcfile on /kaggle/working type ext4 (rw,relatime)
/tmp/fs/kaggle_YBCZdHDIZmW2XvYn0QjNsAjbAtYqr3QY2nZifZ60iVY-356678632-webtier/srcfile on /kaggle/lib type ext4 (ro,relatime)
/tmp/fs/kaggle_YBCZdHDIZmW2XvYn0QjNsAjbAtYqr3QY2nZifZ60iVY-356678632-webtier/srcfile on /kaggle/input type ext4 (ro,relatime)
192.168.16.2:/data/kagglesdsdata/datasets/1216995/2032605/dlzksj5bjysi on /kaggle/input/datasets/xthink/coco-2017-val-images type nfs (ro,relatime,vers=3,rsize=524288,wsize=524288,namlen=255,hard,nolock,fatal_neterrors=none,proto=tcp,timeo=600,retrans=2,sec=sys,mountaddr=192.168.16.2,mountvers=3,mountport=2050,mountproto=tcp,local_lock=all,addr=192.168.16.2)
/tmp/fs/kaggle_YBCZdHDIZmW2XvYn0QjNsAjbAtYqr3QY2nZifZ60iVY-356678632-webtier/srcfile on /etc/secrets/kaggle type ext4 (ro,relatime)


Input directory details:
total 12
drwxr-xr-x 3 root root 4096 Oct  9 07:3

In [9]:
from pathlib import Path

DATASET_ROOT = Path("/kaggle/input/datasets/xthink/coco-2017-val-images")

print("Dataset contents:")
for p in DATASET_ROOT.iterdir():
    print(p.name, "—", "folder" if p.is_dir() else "file")

print("\nSearching for image files...")
image_paths = sorted(
    p for p in DATASET_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

print("Total images found:", len(image_paths))
print("First 5 images:")
for p in image_paths[:5]:
    print(p)

Dataset contents:
val2017 — folder

Searching for image files...
Total images found: 5000
First 5 images:
/kaggle/input/datasets/xthink/coco-2017-val-images/val2017/000000000139.jpg
/kaggle/input/datasets/xthink/coco-2017-val-images/val2017/000000000285.jpg
/kaggle/input/datasets/xthink/coco-2017-val-images/val2017/000000000632.jpg
/kaggle/input/datasets/xthink/coco-2017-val-images/val2017/000000000724.jpg
/kaggle/input/datasets/xthink/coco-2017-val-images/val2017/000000000776.jpg


In [10]:
from pathlib import Path
from PIL import Image
import numpy as np
import csv

WORK = Path("/kaggle/working/ssd_project")
PROCESSED = WORK / "processed"
DATA = WORK / "data"

DATA.mkdir(parents=True, exist_ok=True)

# Confirm preprocessing completed
processed_images = sorted(PROCESSED.glob("*.png"))
print("Processed images found:", len(processed_images))
assert len(processed_images) == 5000

# Image and template dimensions
W = H = 256
TW = TH = 32
N = 100  # Start with 100 images for testing

rng = np.random.default_rng(42)

# Create a random template shared by all images
template = rng.integers(0, 256, size=(TH, TW), dtype=np.uint8)
template.tofile(DATA / "template.bin")

# Create test dataset
dataset_dir = DATA / f"set_{N}"
dataset_dir.mkdir(parents=True, exist_ok=True)

ground_truth = []

for idx in range(N):
    with Image.open(PROCESSED / f"{idx:05d}.png") as im:
        img = np.array(im.convert("L"), dtype=np.uint8)

    # Plant the template in approximately 30% of images
    if rng.random() < 0.30:
        x = int(rng.integers(0, W - TW + 1))
        y = int(rng.integers(0, H - TH + 1))
        img[y:y+TH, x:x+TW] = template
        planted = 1
    else:
        x = y = -1
        planted = 0

    img.tofile(dataset_dir / f"{idx:06d}.bin")
    ground_truth.append([idx, planted, x, y])

with open(dataset_dir / "ground_truth.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["image_id", "planted", "x", "y"])
    writer.writerows(ground_truth)

print("Template saved:", DATA / "template.bin")
print("Binary test images:", len(list(dataset_dir.glob("*.bin"))))
print("Ground-truth CSV saved:", dataset_dir / "ground_truth.csv")

Processed images found: 5000
Template saved: /kaggle/working/ssd_project/data/template.bin
Binary test images: 100
Ground-truth CSV saved: /kaggle/working/ssd_project/data/set_100/ground_truth.csv


In [11]:
%%writefile /kaggle/working/ssd_project/ssd_seq.c
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <inttypes.h>
#include <limits.h>
#include <time.h>

#define W 256
#define H 256
#define TW 32
#define TH 32
#define PIXELS (W * H)
#define TP (TW * TH)

typedef struct {
    int x;
    int y;
    int64_t score;
} Result;

static double wall_time(void) {
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + (double)ts.tv_nsec / 1e9;
}

static Result match_ssd(const uint8_t *image, const uint8_t *T) {
    Result best = {-1, -1, INT64_MAX};

    for (int y = 0; y <= H - TH; y++) {
        for (int x = 0; x <= W - TW; x++) {
            int64_t score = 0;

            for (int i = 0; i < TH; i++) {
                for (int j = 0; j < TW; j++) {
                    int a = image[(y + i) * W + (x + j)];
                    int b = T[i * TW + j];
                    int diff = a - b;
                    score += (int64_t)diff * diff;
                }
            }

            if (score < best.score) {
                best.score = score;
                best.x = x;
                best.y = y;
            }
        }
    }

    return best;
}

int main(int argc, char **argv) {
    if (argc != 4) {
        fprintf(stderr, "Usage: %s DATA_DIR N OUTPUT.csv\n", argv[0]);
        return 1;
    }

    const char *dir = argv[1];
    int N = atoi(argv[2]);
    const char *outpath = argv[3];

    if (N <= 0) {
        fprintf(stderr, "N must be positive.\n");
        return 1;
    }

    uint8_t T[TP];

    FILE *tf = fopen("/kaggle/working/ssd_project/data/template.bin", "rb");
    if (!tf || fread(T, 1, TP, tf) != TP) {
        fprintf(stderr, "Cannot read template.bin\n");
        if (tf) fclose(tf);
        return 1;
    }
    fclose(tf);

    FILE *out = fopen(outpath, "w");
    if (!out) {
        perror("Cannot create output CSV");
        return 1;
    }

    fprintf(out, "image_id,x,y,ssd\n");

    double start = wall_time();

    for (int k = 0; k < N; k++) {
        char path[512];
        snprintf(path, sizeof(path), "%s/%06d.bin", dir, k);

        FILE *f = fopen(path, "rb");
        if (!f) {
            fprintf(stderr, "Cannot open %s\n", path);
            fclose(out);
            return 1;
        }

        uint8_t *img = malloc(PIXELS);
        if (!img || fread(img, 1, PIXELS, f) != PIXELS) {
            fprintf(stderr, "Cannot read image %s\n", path);
            free(img);
            fclose(f);
            fclose(out);
            return 1;
        }

        fclose(f);

        Result r = match_ssd(img, T);

        fprintf(out, "%d,%d,%d,%" PRId64 "\n",
                k, r.x, r.y, r.score);

        free(img);
    }

    double elapsed = wall_time() - start;
    fclose(out);

    printf("Sequential wall time: %.6f seconds\n", elapsed);
    printf("Results saved to: %s\n", outpath);

    return 0;
}

Writing /kaggle/working/ssd_project/ssd_seq.c


In [12]:
from pathlib import Path

WORK = Path("/kaggle/working/ssd_project")
c_file = WORK / "ssd_seq.c"

source = c_file.read_text()

if not source.startswith("#define _POSIX_C_SOURCE"):
    source = "#define _POSIX_C_SOURCE 200809L\n" + source

c_file.write_text(source)
print("Timing declaration fixed.")

Timing declaration fixed.


In [13]:
import subprocess

result = subprocess.run(
    [
        "gcc", "-O3", "-std=c11",
        str(WORK / "ssd_seq.c"),
        "-o", str(WORK / "ssd_seq")
    ],
    capture_output=True, text=True
)

print("Return code:", result.returncode)
print(result.stderr or "Compilation successful!")
assert result.returncode == 0

Return code: 0
Compilation successful!


In [14]:
cmd = [
    str(WORK / "ssd_seq"),
    str(WORK / "data" / "set_100"),
    "100",
    str(WORK / "sequential_100.csv")
]

result = subprocess.run(cmd, capture_output=True, text=True)

print(result.stdout)
print(result.stderr)
assert result.returncode == 0

Sequential wall time: 2.220387 seconds
Results saved to: /kaggle/working/ssd_project/sequential_100.csv




In [21]:

from pathlib import Path
import subprocess

c_file = WORK / "ssd_seq.c"
executable = WORK / "ssd_omp"

result = subprocess.run(
    [
        "gcc", "-O3", "-std=c11", "-fopenmp",
        str(c_file),
        "-o", str(executable)
    ],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print(result.stderr or "Compilation successful!")
assert result.returncode == 0

Return code: 0
Compilation successful!


In [22]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if file.endswith(".c"):
            print(os.path.join(root, file))

/kaggle/working/test_omp.c
/kaggle/working/ssd_project/ssd_seq.c


In [25]:

import pandas as pd
import numpy as np

seq = pd.read_csv(WORK / "sequential_100.csv")
omp = pd.read_csv(WORK / "ssd_output.csv")

print("Sequential rows:", len(seq))
print("OpenMP rows:", len(omp))

assert len(seq) == 100, "Sequential CSV must contain 100 rows"
assert len(omp) == 100, "OpenMP CSV must contain 100 rows"

for col in ["image_id", "x", "y", "ssd"]:
    assert np.array_equal(seq[col].to_numpy(), omp[col].to_numpy()), (
        f"Mismatch in {col}"
    )

print("PASS: Sequential and OpenMP results match exactly.")
display(seq.head())

Sequential rows: 100
OpenMP rows: 0


AssertionError: OpenMP CSV must contain 100 rows

In [26]:
gt = pd.read_csv(WORK / "data" / "set_100" / "ground_truth.csv")
check = gt.merge(seq, on="image_id", suffixes=("_true", "_found"))

planted = check[check["planted"] == 1]

coordinate_match = (
    (planted["x_true"] == planted["x_found"]) &
    (planted["y_true"] == planted["y_found"])
)

print("Planted images:", len(planted))
print("Correct coordinates:", int(coordinate_match.sum()))
print("Zero-SSD matches:", int((planted["ssd"] == 0).sum()))

assert coordinate_match.all()
assert (planted["ssd"] == 0).all()

print("PASS: All planted templates were found.")

Planted images: 31
Correct coordinates: 31
Zero-SSD matches: 31
PASS: All planted templates were found.


In [28]:
from pathlib import Path

WORK = Path("/kaggle/working/ssd_project")

checks = {
    "Template": WORK / "data/template.bin",
    "Sequential C source": WORK / "ssd_seq.c",
    "Sequential executable": WORK / "ssd_seq",
    "Sequential results": WORK / "sequential_100.csv",
    "OpenMP C source": WORK / "ssd_omp.c",
    "OpenMP executable": WORK / "ssd_omp",
    "OpenMP results": WORK / "openmp_100.csv",
}

for name, path in checks.items():
    if path.is_file():
        print(f"FOUND: {name}")
    else:
        print(f"MISSING: {name}")

print("Test images:", len(list((WORK / "data/set_100").glob("*.bin"))))

FOUND: Template
FOUND: Sequential C source
FOUND: Sequential executable
FOUND: Sequential results
MISSING: OpenMP C source
FOUND: OpenMP executable
MISSING: OpenMP results
Test images: 100


In [29]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

WORK = Path("/kaggle/working/ssd_project")
PROCESSED = WORK / "processed"

images = sorted(PROCESSED.glob("*.png"))
assert len(images) > 0, f"No processed PNG images found in {PROCESSED}"

train_images, test_images = train_test_split(
    [str(p) for p in images],
    test_size=0.20,
    random_state=42
)

pd.DataFrame({"image_path": train_images}).to_csv(
    WORK / "train_split.csv", index=False
)
pd.DataFrame({"image_path": test_images}).to_csv(
    WORK / "test_split.csv", index=False
)

print("Training images:", len(train_images))
print("Testing images:", len(test_images))
print(f"Training percentage: {len(train_images)/len(images)*100:.1f}%")
print(f"Testing percentage: {len(test_images)/len(images)*100:.1f}%")
print("Split files saved in:", WORK)

Training images: 4000
Testing images: 1000
Training percentage: 80.0%
Testing percentage: 20.0%
Split files saved in: /kaggle/working/ssd_project


In [30]:
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/ssd_project")
SOURCE = WORK / "ssd_seq.c"
EXECUTABLE = WORK / "ssd_seq"
DATA = WORK / "data" / "set_100"
OUTPUT = WORK / "sequential_100.csv"

assert SOURCE.exists(), f"Missing source: {SOURCE}"
assert (WORK / "data" / "template.bin").exists(), "Missing SSD template"
assert len(list(DATA.glob("*.bin"))) == 100, "Expected 100 test images"

# Compile
result = subprocess.run(
    ["gcc", "-O3", "-std=c11", str(SOURCE), "-o", str(EXECUTABLE)],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Sequential SSD compilation failed")

print("Sequential SSD compiled successfully.")

# Run
result = subprocess.run(
    [str(EXECUTABLE), str(DATA), "100", str(OUTPUT)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Sequential SSD execution failed")

print("Results exist:", OUTPUT.exists())

Sequential SSD compiled successfully.
Sequential wall time: 2.263616 seconds
Results saved to: /kaggle/working/ssd_project/sequential_100.csv

Results exist: True


In [32]:
from pathlib import Path

WORK = Path("/kaggle/working/ssd_project")

print("Project folder exists:", WORK.exists())

if WORK.exists():
    print("\nFiles in project folder:")
    for p in WORK.rglob("*"):
        if p.is_file() and (
            p.name.endswith(".c") or p.name in ["ssd_seq", "ssd_omp"]
        ):
            print(p)
else:
    print("Project folder not found.")

Project folder exists: True

Files in project folder:
/kaggle/working/ssd_project/ssd_seq.c
/kaggle/working/ssd_project/ssd_seq
/kaggle/working/ssd_project/ssd_omp


In [35]:
from pathlib import Path
import subprocess

WORK = Path("/kaggle/working/ssd_project")
DATA = WORK / "data" / "set_100"
EXECUTABLE = WORK / "ssd_omp"
OUTPUT = WORK / "openmp_100.csv"

result = subprocess.run(
    [str(EXECUTABLE), str(DATA), "100", str(OUTPUT)],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("Output:", result.stdout)
print("Errors:", result.stderr)
print("CSV created:", OUTPUT.exists())

if result.returncode != 0:
    raise RuntimeError("OpenMP execution failed.")

Return code: 0
Output: Sequential wall time: 2.247461 seconds
Results saved to: /kaggle/working/ssd_project/openmp_100.csv

Errors: 
CSV created: True


In [36]:
import pandas as pd
import numpy as np

WORK = Path("/kaggle/working/ssd_project")

seq = pd.read_csv(WORK / "sequential_100.csv")
omp = pd.read_csv(WORK / "openmp_100.csv")

assert len(seq) == 100 and len(omp) == 100

for col in ["image_id", "x", "y", "ssd"]:
    assert np.array_equal(seq[col].to_numpy(), omp[col].to_numpy()), (
        f"Mismatch in column: {col}"
    )

print("PASS: Sequential and OpenMP results match for all 100 images.")
print(seq.head())

PASS: Sequential and OpenMP results match for all 100 images.
   image_id    x    y      ssd
0         0   39  132        0
1         1   67  208        0
2         2  165  215  5886668
3         3  203  168  6085806
4         4   16   67  5518843


In [37]:
gt = pd.read_csv(WORK / "data" / "set_100" / "ground_truth.csv")
results = pd.read_csv(WORK / "sequential_100.csv")

check = gt.merge(results, on="image_id", suffixes=("_true", "_found"))
planted = check[check["planted"] == 1].copy()

correct = (
    (planted["x_true"] == planted["x_found"]) &
    (planted["y_true"] == planted["y_found"]) &
    (planted["ssd"] == 0)
)

print("Images with planted template:", len(planted))
print("Correct exact matches:", int(correct.sum()))
print("Planted-template accuracy:",
      f"{100 * correct.mean():.2f}%" if len(planted) else "N/A")

Images with planted template: 31
Correct exact matches: 31
Planted-template accuracy: 100.00%


In [38]:

from pathlib import Path

for f in sorted(Path("/kaggle/working/ssd_project").iterdir()):
    print(f.name)

data
openmp_100.csv
processed
sequential_100.csv
ssd_omp
ssd_output.csv
ssd_seq
ssd_seq.c
test_split.csv
train_split.csv


In [41]:

from pathlib import Path

WORK = Path("/kaggle/working/ssd_project")

print("Project directory exists:", WORK.exists())
print("\nFiles inside project directory:")

for f in sorted(WORK.rglob("*")):
    if f.is_file():
        print(f.relative_to(WORK))

Project directory exists: True

Files inside project directory:
data/set_100/000000.bin
data/set_100/000001.bin
data/set_100/000002.bin
data/set_100/000003.bin
data/set_100/000004.bin
data/set_100/000005.bin
data/set_100/000006.bin
data/set_100/000007.bin
data/set_100/000008.bin
data/set_100/000009.bin
data/set_100/000010.bin
data/set_100/000011.bin
data/set_100/000012.bin
data/set_100/000013.bin
data/set_100/000014.bin
data/set_100/000015.bin
data/set_100/000016.bin
data/set_100/000017.bin
data/set_100/000018.bin
data/set_100/000019.bin
data/set_100/000020.bin
data/set_100/000021.bin
data/set_100/000022.bin
data/set_100/000023.bin
data/set_100/000024.bin
data/set_100/000025.bin
data/set_100/000026.bin
data/set_100/000027.bin
data/set_100/000028.bin
data/set_100/000029.bin
data/set_100/000030.bin
data/set_100/000031.bin
data/set_100/000032.bin
data/set_100/000033.bin
data/set_100/000034.bin
data/set_100/000035.bin
data/set_100/000036.bin
data/set_100/000037.bin
data/set_100/000038.bin


In [42]:

from pathlib import Path
import zipfile

WORK = Path("/kaggle/working/ssd_project")
ZIP_PATH = Path("/kaggle/working/ssd_project_files.zip")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for f in WORK.rglob("*"):
        if not f.is_file():
            continue

        # Skip dataset folders to avoid packaging large data
        if "data" in f.relative_to(WORK).parts:
            continue

        if "processed" in f.relative_to(WORK).parts:
            continue

        z.write(f, f.relative_to(WORK))
        print("Added:", f.relative_to(WORK))

print("\nZIP exists:", ZIP_PATH.exists())
print("ZIP size (bytes):", ZIP_PATH.stat().st_size)

Added: test_split.csv
Added: ssd_seq.c
Added: ssd_seq
Added: sequential_100.csv
Added: ssd_omp
Added: train_split.csv
Added: openmp_100.csv
Added: ssd_output.csv

ZIP exists: True
ZIP size (bytes): 26945


In [43]:

from IPython.display import FileLink, display

display(FileLink("/kaggle/working/ssd_project_files.zip"))

/kaggle/working/ssd_project_files.zip

In [44]:

from pathlib import Path

omp_source = Path("/kaggle/working/test_omp.c")

print(omp_source.read_text())

#include <stdio.h>
#include <omp.h>

int main(void) {
    #pragma omp parallel
    {
        printf("Hello from thread %d\n", omp_get_thread_num());
    }
    return 0;
}



In [45]:

import shutil

src = Path("/kaggle/working/test_omp.c")
dst = Path("/kaggle/working/ssd_project/ssd_omp.c")

shutil.copy2(src, dst)

print("Copied:", dst)
print("File exists:", dst.exists())

Copied: /kaggle/working/ssd_project/ssd_omp.c
File exists: True


In [46]:
import pandas as pd

print("TRAIN SPLIT")
display(pd.read_csv(WORK / "train_split.csv").head())

print("TEST SPLIT")
display(pd.read_csv(WORK / "test_split.csv").head())

TRAIN SPLIT


,image_path
0,/kaggle/working/ssd_project/processed/04227.png
1,/kaggle/working/ssd_project/processed/04676.png
2,/kaggle/working/ssd_project/processed/00800.png
3,/kaggle/working/ssd_project/processed/03671.png
4,/kaggle/working/ssd_project/processed/04193.png


TEST SPLIT


,image_path
0,/kaggle/working/ssd_project/processed/01501.png
1,/kaggle/working/ssd_project/processed/02586.png
2,/kaggle/working/ssd_project/processed/02653.png
3,/kaggle/working/ssd_project/processed/01055.png
4,/kaggle/working/ssd_project/processed/00705.png


In [47]:

# Search the current notebook's saved source, if available.
import os

for root, dirs, files in os.walk("/kaggle/working"):
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]
    for file in files:
        if file.endswith((".py", ".ipynb", ".sh", ".c")):
            path = os.path.join(root, file)
            try:
                text = open(path, encoding="utf-8").read()
                if "ssd_omp" in text or "omp parallel" in text:
                    print("\n---", path, "---")
                    print(text[:12000])
            except (UnicodeDecodeError, OSError):
                pass


--- /kaggle/working/test_omp.c ---
#include <stdio.h>
#include <omp.h>

int main(void) {
    #pragma omp parallel
    {
        printf("Hello from thread %d\n", omp_get_thread_num());
    }
    return 0;
}


--- /kaggle/working/ssd_project/ssd_omp.c ---
#include <stdio.h>
#include <omp.h>

int main(void) {
    #pragma omp parallel
    {
        printf("Hello from thread %d\n", omp_get_thread_num());
    }
    return 0;
}


--- /kaggle/working/.virtual_documents/__notebook_source__.ipynb ---
get_ipython().getoutput("gcc --version")
get_ipython().getoutput("nproc")
get_ipython().getoutput("ls -lah /kaggle/input/")


import os

for root, dirs, files in os.walk("/kaggle/input"):
    images = [
        f for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    if images:
        print("Folder:", root)
        print("Image count in folder:", len(images))
        print("Sample files:", images[:5])


get_ipython().run_cell_magic("writefile", " /kaggle/working/test_om

In [48]:

import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/ssd_project")

# Copy the updated ssd_omp.c from GitHub into Kaggle first,
# or replace this source file with the committed code.

compile_result = subprocess.run(
    [
        "gcc", "-O3", "-std=c11", "-fopenmp",
        str(WORK / "ssd_omp.c"),
        "-o", str(WORK / "ssd_omp")
    ],
    capture_output=True,
    text=True
)

print(compile_result.stderr or "Compilation successful!")
assert compile_result.returncode == 0

run_result = subprocess.run(
    [
        str(WORK / "ssd_omp"),
        str(WORK / "data" / "set_100"),
        "100",
        str(WORK / "openmp_100.csv")
    ],
    capture_output=True,
    text=True,
    env={**__import__("os").environ, "OMP_NUM_THREADS": "4"}
)

print(run_result.stdout)
print(run_result.stderr)
assert run_result.returncode == 0

Compilation successful!
Hello from thread 2
Hello from thread 0
Hello from thread 1
Hello from thread 3




In [49]:
from pathlib import Path

p = Path("/kaggle/working/ssd_project/ssd_omp.c")
print(p.read_text())

#include <stdio.h>
#include <omp.h>

int main(void) {
    #pragma omp parallel
    {
        printf("Hello from thread %d\n", omp_get_thread_num());
    }
    return 0;
}



In [51]:

from pathlib import Path

code = r'''
#define _POSIX_C_SOURCE 200809L

#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <inttypes.h>
#include <limits.h>
#include <time.h>
#include <omp.h>

#define W 256
#define H 256
#define TW 32
#define TH 32
#define PIXELS (W * H)
#define TP (TW * TH)

typedef struct {
    int x;
    int y;
    int64_t score;
} Result;

static double wall_time(void) {
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + (double)ts.tv_nsec / 1e9;
}

static Result match_ssd(const uint8_t *image, const uint8_t *T) {
    Result best = {-1, -1, INT64_MAX};

    for (int y = 0; y <= H - TH; y++) {
        for (int x = 0; x <= W - TW; x++) {
            int64_t score = 0;

            for (int i = 0; i < TH; i++) {
                for (int j = 0; j < TW; j++) {
                    int a = image[(y + i) * W + (x + j)];
                    int b = T[i * TW + j];
                    int diff = a - b;
                    score += (int64_t)diff * diff;
                }
            }

            if (score < best.score) {
                best.score = score;
                best.x = x;
                best.y = y;
            }
        }
    }

    return best;
}

int main(int argc, char **argv) {
    if (argc != 4) {
        fprintf(stderr, "Usage: %s DATA_DIR N OUTPUT.csv\n", argv[0]);
        return 1;
    }

    const char *dir = argv[1];
    int N = atoi(argv[2]);
    const char *outpath = argv[3];

    if (N <= 0) {
        fprintf(stderr, "N must be positive.\n");
        return 1;
    }

    uint8_t T[TP];
    FILE *tf = fopen(
        "/kaggle/working/ssd_project/data/template.bin", "rb"
    );

    if (!tf || fread(T, 1, TP, tf) != TP) {
        fprintf(stderr, "Cannot read template.bin\n");
        if (tf) fclose(tf);
        return 1;
    }
    fclose(tf);

    Result *results = calloc((size_t)N, sizeof(Result));
    if (!results) {
        perror("Cannot allocate results");
        return 1;
    }

    int failed = 0;
    double start = wall_time();

    #pragma omp parallel for schedule(static) reduction(|:failed)
    for (int k = 0; k < N; k++) {
        char path[512];
        snprintf(path, sizeof(path), "%s/%06d.bin", dir, k);

        FILE *f = fopen(path, "rb");
        if (!f) {
            #pragma omp critical
            fprintf(stderr, "Cannot open %s\n", path);
            failed = 1;
            continue;
        }

        uint8_t *img = malloc(PIXELS);
        if (!img || fread(img, 1, PIXELS, f) != PIXELS) {
            #pragma omp critical
            fprintf(stderr, "Cannot read image %s\n", path);
            free(img);
            fclose(f);
            failed = 1;
            continue;
        }

        fclose(f);
        results[k] = match_ssd(img, T);
        free(img);
    }

    if (failed) {
        free(results);
        return 1;
    }

    FILE *out = fopen(outpath, "w");
    if (!out) {
        perror("Cannot create output CSV");
        free(results);
        return 1;
    }

    fprintf(out, "image_id,x,y,ssd\n");

    for (int k = 0; k < N; k++) {
        fprintf(out, "%d,%d,%d,%" PRId64 "\n",
                k, results[k].x, results[k].y, results[k].score);
    }

    fclose(out);
    free(results);

    printf("OpenMP threads: %d\n", omp_get_max_threads());
    printf("OpenMP wall time: %.6f seconds\n", wall_time() - start);
    printf("Results saved to: %s\n", outpath);

    return 0;
}
'''

target = Path("/kaggle/working/ssd_project/ssd_omp.c")
target.write_text(code.strip() + "\n")

print("Updated:", target)
print("File size:", target.stat().st_size, "bytes")
print("OpenMP parallel loop present:", "#pragma omp parallel for" in target.read_text())

Updated: /kaggle/working/ssd_project/ssd_omp.c
File size: 3428 bytes
OpenMP parallel loop present: True


In [52]:

import subprocess
import os

work = "/kaggle/working/ssd_project"

compile_result = subprocess.run(
    ["gcc", "-O3", "-std=c11", "-fopenmp",
     f"{work}/ssd_omp.c", "-o", f"{work}/ssd_omp"],
    capture_output=True, text=True
)

print(compile_result.stderr or "Compilation successful!")
assert compile_result.returncode == 0

env = os.environ.copy()
env["OMP_NUM_THREADS"] = "4"

run_result = subprocess.run(
    [f"{work}/ssd_omp", f"{work}/data/set_100", "100",
     f"{work}/openmp_100.csv"],
    capture_output=True, text=True, env=env
)

print(run_result.stdout)
if run_result.stderr:
    print(run_result.stderr)

assert run_result.returncode == 0

Compilation successful!
OpenMP threads: 4
OpenMP wall time: 1.109093 seconds
Results saved to: /kaggle/working/ssd_project/openmp_100.csv

